In [16]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [17]:
import pandas as pd
import numpy as np

# Fama-French 3 factor model 회귀를 위해 팩터 데이터 로드
factors = pd.read_csv("../../00_input/Factors.csv", index_col=0, parse_dates=True)  # 팩터(MKT, SMB, HML, MOM, RF)

In [18]:
# 전처리 함수 정의
def make_ff_factors(factors, annual_rf=True):
    """
    factors: DataFrame with columns ['KOSPI','SMB','HML','MOM','RF']
    """
    
    df = factors.copy()

    # 0. resampling
    df = df.resample('ME').last()
    
    # 1. 지수 → 수익률 변환
    ret_cols = ['KOSPI','SMB','HML','MOM']
    df[ret_cols] = df[ret_cols].pct_change()
    
    # 2. 무위험금리 변환 (연율 → 일/월 수익률)
    df['RF'] = df['RF'] / 100  # % → 소수화 (예: 3.5% → 0.035)
    df['RF'] = (1 + df['RF']) ** (1/12) - 1
    
    # 4. 컬럼 정리
    df = df[['KOSPI','SMB','HML','MOM','RF']].dropna()
    
    return df

In [19]:
# 함수 실행
factors_monthly = make_ff_factors(factors)[:-1]

In [20]:
# 수정주가 데이터 로드
adj_close        = pd.read_csv("../../00_input/KOSPI_KOSDAQ_adj_close.csv", index_col=0, parse_dates=True)         # 수정주가

In [21]:
# 월간 수익률 계산
return_monthly   = adj_close.resample('ME').last().pct_change(fill_method=None)         # 수정주가 데이터로 월별 수익률 계산
return_monthly   = return_monthly.reindex(factors_monthly.index)                        # 팩터 데이터와 인덱스 통일

In [22]:
# 수익률 분해를 위한 팩터 정리 - 종속변수 설정(Ri - RF)
excess_return = return_monthly.sub(factors_monthly['RF'], axis=0)

In [23]:
# 수익률 분해를 위한 팩터 정리 - 독립변수 설정
factors_monthly['MKT'] = factors_monthly['KOSPI'] - factors_monthly['RF']            # MKT 팩터(시장 포트폴리오 초과 수익률)
X = factors_monthly[['MKT']].copy()

In [24]:
def rolling_capm_beta(y: pd.Series,
                      X: pd.DataFrame,
                      window: int = 12) -> pd.Series:
    df = pd.concat([y.rename("y"), X[["MKT"]]], axis=1).dropna()
    beta = pd.Series(index=y.index, dtype=float)

    for t in range(len(df)):
        end_idx = df.index[t]
        start_pos = t - window + 1
        if start_pos < 0:
            continue

        train = df.iloc[start_pos:t+1]
        y_win = train["y"]
        X_win = sm.add_constant(train[["MKT"]], has_constant="add")
        model = sm.OLS(y_win, X_win).fit()

        beta.loc[end_idx] = model.params["MKT"]

    return beta

In [25]:
from joblib import Parallel, delayed
from tqdm import tqdm

tickers = excess_return.columns.tolist()

# 병렬로 iVOL 계산
beta_list = Parallel(n_jobs=-1)(
    delayed(rolling_capm_beta)(excess_return[ticker], X, window=12)
    for ticker in tqdm(tickers, desc="Calculating Beta")
)

beta_df = pd.concat(beta_list, axis=1)
beta_df.columns = tickers



























































































































































































































































































































































































































































































































































































































Calculating Beta: 100%|██████████| 3948/3948 [04:31<00:00, 14.52it/s] 


In [27]:
beta_df.to_csv("./output/beta.csv", index=True, encoding="utf-8-sig")

In [28]:
from pathlib import Path
from scipy.stats import norm

In [29]:
class GaussianHMM2State:
    """
    2-state Gaussian HMM for monthly regime detection.

    State ordering is normalized after fitting so that the higher-mean
    state is treated as the bull regime.
    """

    def __init__(self, n_iter: int = 100, tol: float = 1e-4, random_state: int = 42):
        self.n_iter = n_iter
        self.tol = tol
        self.random_state = random_state
        self.pi = None
        self.A = None
        self.mu = None
        self.sigma = None
        self.bull_state = 1

    def _emission_prob(self, x: np.ndarray) -> np.ndarray:
        B = np.zeros((len(x), 2))
        for k in range(2):
            B[:, k] = norm.pdf(x, loc=self.mu[k], scale=self.sigma[k])
        return np.clip(B, 1e-300, None)

    def _forward(self, B: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        T = B.shape[0]
        alpha_hat = np.zeros((T, 2))
        c = np.zeros(T)
        alpha_hat[0] = self.pi * B[0]
        c[0] = alpha_hat[0].sum()
        alpha_hat[0] /= c[0] + 1e-300
        for t in range(1, T):
            alpha_hat[t] = (alpha_hat[t - 1] @ self.A) * B[t]
            c[t] = alpha_hat[t].sum()
            alpha_hat[t] /= c[t] + 1e-300
        return alpha_hat, c

    def _backward(self, B: np.ndarray, c: np.ndarray) -> np.ndarray:
        T = B.shape[0]
        beta_hat = np.zeros((T, 2))
        beta_hat[-1] = 1.0 / (c[-1] + 1e-300)
        for t in range(T - 2, -1, -1):
            beta_hat[t] = (self.A @ (B[t + 1] * beta_hat[t + 1])) / (c[t] + 1e-300)
        return beta_hat

    def fit(self, x: np.ndarray) -> "GaussianHMM2State":
        x = np.asarray(x, dtype=float)
        T = len(x)
        if T < 2:
            raise ValueError('Need at least 2 observations to fit HMM.')
        sorted_x = np.sort(x)
        self.mu = np.array([
            sorted_x[: T // 2].mean(),
            sorted_x[T // 2 :].mean(),
        ])
        self.sigma = np.array([
            max(sorted_x[: T // 2].std(), 1e-6),
            max(sorted_x[T // 2 :].std(), 1e-6),
        ])
        self.pi = np.array([0.5, 0.5])
        self.A = np.array([[0.95, 0.05], [0.05, 0.95]])
        prev_loglik = -np.inf
        for _ in range(self.n_iter):
            B = self._emission_prob(x)
            alpha_hat, c = self._forward(B)
            beta_hat = self._backward(B, c)
            gamma = alpha_hat * beta_hat
            gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
            xi = np.zeros((T - 1, 2, 2))
            for t in range(T - 1):
                xi[t] = (
                    alpha_hat[t, :, None]
                    * self.A
                    * B[t + 1, None, :]
                    * beta_hat[t + 1, None, :]
                )
                xi[t] /= xi[t].sum() + 1e-300
            self.pi = gamma[0]
            self.A = xi.sum(axis=0) / (gamma[:-1].sum(axis=0)[:, None] + 1e-300)
            self.A = self.A / self.A.sum(axis=1, keepdims=True)
            for k in range(2):
                w = gamma[:, k]
                self.mu[k] = (w * x).sum() / (w.sum() + 1e-300)
                self.sigma[k] = np.sqrt((w * (x - self.mu[k]) ** 2).sum() / (w.sum() + 1e-300))
                self.sigma[k] = max(self.sigma[k], 1e-6)
            loglik = np.log(c + 1e-300).sum()
            if abs(loglik - prev_loglik) < self.tol:
                break
            prev_loglik = loglik
        self.bull_state = int(np.argmax(self.mu))
        return self

    def filtered_probs(self, x: np.ndarray) -> np.ndarray:
        B = self._emission_prob(np.asarray(x, dtype=float))
        alpha_hat, _ = self._forward(B)
        return alpha_hat

    def bull_prob(self, x: np.ndarray) -> np.ndarray:
        probs = self.filtered_probs(x)
        return probs[:, self.bull_state]


def smooth_regime(bull_prob: np.ndarray, window: int = 2) -> np.ndarray:
    return pd.Series(bull_prob).rolling(window=window, min_periods=1).mean().to_numpy()


def rolling_hmm_regime(
    market_excess: pd.Series,
    min_train: int = 36,
    rolling_window: int | None = None,
    smooth_window: int = 2,
    n_iter: int = 100,
    tol: float = 1e-4,
    random_state: int = 42,
) -> pd.DataFrame:
    series = market_excess.dropna().astype(float)
    out = pd.DataFrame(index=series.index, columns=['bull_prob', 'bull_prob_smoothed', 'regime'])

    for end_pos in range(min_train - 1, len(series)):
        if rolling_window is None:
            train = series.iloc[: end_pos + 1]
        else:
            start_pos = max(0, end_pos + 1 - rolling_window)
            train = series.iloc[start_pos : end_pos + 1]

        if len(train) < min_train:
            continue

        hmm = GaussianHMM2State(
            n_iter=n_iter,
            tol=tol,
            random_state=random_state,
        ).fit(train.to_numpy())

        bull_prob_hist = hmm.bull_prob(train.to_numpy())
        bull_prob_smoothed = smooth_regime(bull_prob_hist, window=smooth_window)
        out.loc[train.index[-1], 'bull_prob'] = bull_prob_hist[-1]
        out.loc[train.index[-1], 'bull_prob_smoothed'] = bull_prob_smoothed[-1]
        out.loc[train.index[-1], 'regime'] = 'bull' if bull_prob_smoothed[-1] >= 0.5 else 'bear'

    out['bull_prob'] = pd.to_numeric(out['bull_prob'], errors='coerce')
    out['bull_prob_smoothed'] = pd.to_numeric(out['bull_prob_smoothed'], errors='coerce')
    out['regime_next'] = out['regime'].shift(1)
    out['bull_prob_next'] = out['bull_prob_smoothed'].shift(1)
    return out

In [30]:
regime_df = rolling_hmm_regime(
    market_excess=X['MKT'],
    min_train=36,
    rolling_window=None,
    smooth_window=2,
    n_iter=100,
    tol=1e-4,
    random_state=42,
)

regime_df.tail(12)

,bull_prob,bull_prob_smoothed,regime,regime_next,bull_prob_next
Date,,,,,
2025-03-31,0.993748,0.993706,bull,bull,0.992732
2025-04-30,0.993780,0.993767,bull,bull,0.993706
2025-05-31,0.991563,0.992684,bull,bull,0.993767
2025-06-30,0.893160,0.942281,bull,bull,0.992684
2025-07-31,0.932448,0.914650,bull,bull,0.942281
2025-08-31,0.967255,0.950160,bull,bull,0.914650
2025-09-30,0.966900,0.967555,bull,bull,0.950160
2025-10-31,0.937521,0.497395,bear,bull,0.967555
2025-11-30,0.146231,0.105901,bear,bear,0.497395


In [31]:
output_dir = Path('./output')
output_dir.mkdir(parents=True, exist_ok=True)

regime_df.to_csv(output_dir / 'hmm_regime.csv', index=True, encoding='utf-8-sig')

In [ ]:
total_adj_close = pd.read_csv("../../00_input/KOSPI_KOSDAQ_total_adj_close.csv", index_col=0, parse_dates=True)
mkt_cap_raw = pd.read_csv("../../00_input/KOSPI_KOSDAQ_mkt_cap.csv", index_col=0, parse_dates=True)
trading_value_60_raw = pd.read_csv("../../00_input/KOSPI_KOSDAQ_trading_value_60.csv", index_col=0, parse_dates=True)
trading_value_raw = pd.read_csv("../../00_input/KOSPI_KOSDAQ_trading_value.csv", index_col=0, parse_dates=True)

monthly_rets = total_adj_close.resample("ME").last().pct_change(fill_method=None)
monthly_rets_wins = monthly_rets.clip(
    lower=monthly_rets.quantile(0.01),
    upper=monthly_rets.quantile(0.99),
    axis=1,
)

mkt_cap_monthly = mkt_cap_raw.resample("ME").last()
trading_value_60_monthly = trading_value_60_raw.resample("ME").last()
daily_ret = adj_close.pct_change(fill_method=None)
daily_illiq = daily_ret.abs().div(trading_value_raw).replace([np.inf, -np.inf], np.nan)
illiq = daily_illiq.resample("ME").mean()

In [ ]:
def run_regime_beta_portfolio(
    beta_signal: pd.DataFrame,
    regime_signal: pd.DataFrame,
    monthly_returns: pd.DataFrame,
    rf_monthly: pd.Series,
    trading_value_60: pd.DataFrame,
    mkt_cap: pd.DataFrame,
    illiq: pd.DataFrame,
    n_select: int = 20,
    bull_side: str = "low",
    bear_side: str = "high",
    weight_method: str = "Equal",
    trading_threshold: float = 0.10,
    high_cost: float = 0.008,
    low_cost: float = 0.003,
    illiq_threshold: float = 0.80,
    initial_nav: float = 1.0,
) -> tuple[pd.DataFrame, dict[pd.Timestamp, list[str]]]:
    common_index = beta_signal.index
    for frame in [regime_signal, monthly_returns, trading_value_60, mkt_cap, illiq]:
        common_index = common_index.intersection(frame.index)
    common_index = common_index.intersection(rf_monthly.index).sort_values()

    portfolio_return = pd.Series(index=common_index, dtype=float)
    total_trade = pd.Series(index=common_index, dtype=float)
    invested_weight = pd.Series(index=common_index, dtype=float)
    cash_weight = pd.Series(index=common_index, dtype=float)
    regime_used = pd.Series(index=common_index, dtype=object)
    bull_prob_used = pd.Series(index=common_index, dtype=float)
    prev_portfolio = pd.Series(dtype=float)
    picks = {}

    portfolio_return.iloc[0] = 0.0
    invested_weight.iloc[0] = 0.0
    cash_weight.iloc[0] = 1.0
    nav = initial_nav

    def select_side(regime_name: str) -> str | None:
        if regime_name == "bull":
            return bull_side
        if regime_name == "bear":
            return bear_side
        return None

    for i in range(len(common_index) - 1):
        start_date = common_index[i]
        end_date = common_index[i + 1]

        regime_today = regime_signal.loc[start_date, "regime"] if start_date in regime_signal.index else np.nan
        bull_prob_today = regime_signal.loc[start_date, "bull_prob_smoothed"] if start_date in regime_signal.index else np.nan
        regime_used.loc[end_date] = regime_today
        bull_prob_used.loc[end_date] = bull_prob_today

        trading_today = trading_value_60.loc[start_date].dropna()
        liquid_names = trading_today[trading_today > trading_today.quantile(trading_threshold)].index
        factor_today = beta_signal.loc[start_date, liquid_names].dropna()

        side = select_side(regime_today)
        if side == "low":
            basket = factor_today.nsmallest(n_select).index
        elif side == "high":
            basket = factor_today.nlargest(n_select).index
        else:
            basket = pd.Index([])

        basket = basket.intersection(monthly_returns.columns)
        basket = basket.intersection(mkt_cap.columns)

        if len(basket) == 0:
            target_weights = pd.Series(dtype=float)
        elif weight_method == "Cap":
            cap = mkt_cap.loc[start_date, basket].dropna()
            cap = cap[cap > 0]
            target_weights = cap / cap.sum() if not cap.empty else pd.Series(1.0 / len(basket), index=basket)
        else:
            target_weights = pd.Series(1.0 / len(basket), index=basket)

        if prev_portfolio.empty or prev_portfolio.sum() <= 0:
            prev_weights = pd.Series(dtype=float)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        all_index = target_weights.index.union(prev_weights.index)
        target_w = target_weights.reindex(all_index, fill_value=0.0)
        prev_w = prev_weights.reindex(all_index, fill_value=0.0)
        delta_w = target_w - prev_w

        illiq_today = illiq.loc[start_date].dropna()
        illiq_cutoff = illiq_today.quantile(illiq_threshold) if not illiq_today.empty else np.nan
        illiquid_names = illiq_today[illiq_today >= illiq_cutoff].index if pd.notna(illiq_cutoff) else pd.Index([])

        trade_amounts = delta_w.abs() * nav
        cost_rate = np.where(delta_w.index.isin(illiquid_names), high_cost, low_cost)
        trade_cost = float((trade_amounts * cost_rate).sum())

        nav_after_cost = nav - trade_cost
        current_portfolio_value = target_weights * nav_after_cost
        current_cash_value = nav_after_cost * max(0.0, 1.0 - target_weights.sum())

        realized = monthly_returns.loc[end_date, target_weights.index].fillna(0.0)
        next_portfolio_value = current_portfolio_value * (1.0 + realized)
        next_cash_value = current_cash_value * (1.0 + rf_monthly.loc[end_date])
        nav_new = float(next_portfolio_value.sum() + next_cash_value)
        portfolio_ret = nav_new / nav - 1.0

        prev_portfolio = next_portfolio_value
        nav = nav_new

        total_trade.loc[start_date] = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret
        invested_weight.loc[end_date] = target_weights.sum()
        cash_weight.loc[end_date] = max(0.0, 1.0 - target_weights.sum())
        picks[end_date] = list(target_weights.index)

    portfolio_nav = (1.0 + portfolio_return.fillna(0.0)).cumprod() * initial_nav
    portfolio_df = pd.DataFrame(
        {
            "Return": portfolio_return,
            "NAV": portfolio_nav,
            "Trade": total_trade,
            "Regime": regime_used,
            "BullProb": bull_prob_used,
            "InvestedWeight": invested_weight,
            "CashWeight": cash_weight,
        }
    )
    portfolio_df.index.name = "Date"
    portfolio_df.loc[portfolio_df.index[0], "NAV"] = initial_nav
    return portfolio_df, picks

In [ ]:
portfolio_df, picked_stocks = run_regime_beta_portfolio(
    beta_signal=beta_df,
    regime_signal=regime_df,
    monthly_returns=monthly_rets_wins,
    rf_monthly=factors_monthly["RF"],
    trading_value_60=trading_value_60_monthly,
    mkt_cap=mkt_cap_monthly,
    illiq=illiq,
    n_select=20,
    bull_side="low",
    bear_side="high",
    weight_method="Equal",
    trading_threshold=0.10,
    high_cost=0.008,
    low_cost=0.003,
    illiq_threshold=0.80,
    initial_nav=1.0,
)

portfolio_df.tail()

In [ ]:
picked_stocks_df = pd.Series(picked_stocks, name="Basket").to_frame()
picked_stocks_df.index.name = "Date"

portfolio_df.to_csv(output_dir / "regime_beta_portfolio.csv", encoding="utf-8-sig")
picked_stocks_df.to_csv(output_dir / "regime_beta_picks.csv", encoding="utf-8-sig")

In [36]:
def performance_metrics(
    portfolio: pd.DataFrame,
    rf_monthly: pd.Series | None = None,
    periods_per_year: int = 12,
) -> pd.Series:
    returns = portfolio["Return"].dropna()
    nav = portfolio["NAV"].dropna()
    trade = portfolio["Trade"].dropna()

    if len(returns) <= 1 or len(nav) <= 1:
        return pd.Series(dtype=float)

    returns = returns.iloc[1:]
    nav = nav.reindex(returns.index)
    initial_nav = float(portfolio["NAV"].iloc[0])
    final_nav = float(nav.iloc[-1])
    years = len(returns) / periods_per_year

    cagr = (final_nav / initial_nav) ** (1 / years) - 1 if years > 0 else np.nan
    vol_monthly = returns.std()
    vol_annual = vol_monthly * np.sqrt(periods_per_year)

    if rf_monthly is None:
        rf_aligned = pd.Series(0.0, index=returns.index)
    else:
        rf_aligned = rf_monthly.reindex(returns.index).fillna(0.0)

    excess_returns = returns - rf_aligned
    sharpe_ratio = (
        np.sqrt(periods_per_year) * excess_returns.mean() / returns.std()
        if returns.std() > 0 else np.nan
    )

    downside = returns[returns < 0]
    downside_vol = downside.std() * np.sqrt(periods_per_year) if len(downside) > 1 else np.nan
    sortino_ratio = (
        np.sqrt(periods_per_year) * excess_returns.mean() / downside.std()
        if len(downside) > 1 and downside.std() > 0 else np.nan
    )

    cummax_nav = nav.cummax()
    drawdown = nav / cummax_nav - 1
    mdd = drawdown.min()
    calmar_ratio = cagr / abs(mdd) if pd.notna(mdd) and mdd < 0 else np.nan

    monthly_turnover = (trade / nav).replace([np.inf, -np.inf], np.nan)
    avg_turnover = monthly_turnover.mean()
    hit_ratio = (returns > 0).mean()

    return pd.Series(
        {
            "CAGR": cagr,
            "Volatility (ann.)": vol_annual,
            "Sharpe Ratio": sharpe_ratio,
            "Sortino Ratio": sortino_ratio,
            "Calmar Ratio": calmar_ratio,
            "MDD": mdd,
            "Average Turnover (monthly)": avg_turnover,
            "Hit Ratio": hit_ratio,
        }
    )

In [37]:
market_portfolio = pd.DataFrame(index=portfolio_df.index)
market_portfolio["Return"] = factors_monthly["KOSPI"].reindex(portfolio_df.index).fillna(0.0)
market_portfolio["NAV"] = (1.0 + market_portfolio["Return"]).cumprod()
market_portfolio["Trade"] = 0.0

cash_portfolio = pd.DataFrame(index=portfolio_df.index)
cash_portfolio["Return"] = factors_monthly["RF"].reindex(portfolio_df.index).fillna(0.0)
cash_portfolio["NAV"] = (1.0 + cash_portfolio["Return"]).cumprod()
cash_portfolio["Trade"] = 0.0

performance_table = pd.DataFrame(
    {
        "Regime Beta Portfolio": performance_metrics(portfolio_df, rf_monthly=factors_monthly["RF"]),
        "KOSPI": performance_metrics(market_portfolio, rf_monthly=factors_monthly["RF"]),
        "Cash": performance_metrics(cash_portfolio, rf_monthly=factors_monthly["RF"]),
    }
).T

performance_table

,CAGR,Volatility (ann.),Sharpe Ratio,Sortino Ratio,Calmar Ratio,MDD,Average Turnover (monthly),Hit Ratio
Regime Beta Portfolio,-0.144759,0.430196,-0.269309,-0.474245,-0.144997,-0.998359,0.692609,0.463158
KOSPI,0.061857,0.265652,0.178702,0.297652,0.084669,-0.730577,0.000000,0.536842
Cash,0.047642,0.011231,0.000000,NaN,NaN,0.000000,0.000000,1.000000


In [38]:
regime_perf = (
    portfolio_df.dropna(subset=["Return", "Regime"])
    .groupby("Regime")
    .apply(lambda x: performance_metrics(x[["Return", "NAV", "Trade"]], rf_monthly=factors_monthly["RF"]))
)

regime_perf

C:\Users\XH58\AppData\Local\Temp\ipykernel_103572\1623565302.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: performance_metrics(x[["Return", "NAV", "Trade"]], rf_monthly=factors_monthly["RF"]))


,CAGR,Volatility (ann.),Sharpe Ratio,Sortino Ratio,Calmar Ratio,MDD,Average Turnover (monthly),Hit Ratio
Regime,,,,,,,,
bear,-0.427552,0.499375,-0.237129,-0.459221,-0.429695,-0.995012,0.804006,0.419643
bull,-0.240457,0.428799,-0.310932,-0.564526,-0.240852,-0.998359,0.739594,0.402597


In [39]:
performance_table.to_csv(output_dir / "regime_beta_performance.csv", encoding="utf-8-sig")
regime_perf.to_csv(output_dir / "regime_beta_performance_by_regime.csv", encoding="utf-8-sig")